# Day 3: Giving Your Agent a Memory

Welcome to Day 3! So far, we've built agents that can follow instructions and use tools. But they have a major limitation: they have no memory. If you start a new chat, they forget everything you've ever told them. It's like having a conversation with someone who has severe short-term memory loss.

Today, we'll fix that. We'll explore **Context Engineering**: the art and science of giving our agents memory. We'll learn how to make them remember things within a single conversation (short-term memory) and across many different conversations (long-term memory).

**In this notebook, you will learn to:**
- 🧠 Use **Sessions** to give your agent short-term memory for a single conversation.
- ✍️ Use **Session State** to manually save important details like a user's name.
- 📚 Use **Automatic Memory** to give your agent a long-term memory that persists across different chats.
- 🚀 **Upgrade your Universal Knowledge Worker** to remember your preferences.


## Theory Primer: Give it Memory

Yesterday your agent could think and use tools. Today we fix its biggest weakness: **it forgets everything the moment you press Enter.**

### The Goldfish Problem

Large Language Models are **stateless**. That's a fancy way of saying every single API call is a brand-new conversation with a brilliant stranger who has amnesia.

Imagine walking into a bank. You tell the teller, *"My name is Amaka."* You walk out, walk back in, and ask, *"What's my name?"* — and the teller stares at you blankly. That's your agent right now. Not because it's unintelligent, but because nothing was written down.

So how do ChatGPT and Claude appear to remember you? A small trick: **they don't remember — they re-read.** Every time you send a message, the entire conversation so far is quietly sent along with it. Memory isn't magic; it's *paperwork*.

### Messages, Roles, and the Transcript

Think of your conversation as a court transcript. Every line has a speaker label, called a **role**:

- `system` — the standing instructions ("You are a helpful research assistant")
- `user` — what the human said
- `assistant` — what the agent replied

Your job today is to keep this transcript in a list and hand the whole thing back to the model on every turn. Once you do that, follow-up questions like *"Summarise that in Yoruba"* or *"Now compare it to the second option"* suddenly work — because "that" and "the second option" finally have context to point at.

### Sessions: Separate Notebooks for Separate People

A **session** is one continuous conversation, identified by a `session_id`. Think of a doctor's office: every patient has their own file. Ngozi's file never gets mixed with Tunde's. Without sessions, your agent would be like a clinic that keeps one giant folder for the whole town — chaos.

For our Universal Knowledge Worker, sessions mean a user can leave, come back, and pick up exactly where they left off.

### The Context Window (Your Agent's Desk)

Here's the catch: the model can only "see" a limited amount of text at once — the **context window**, measured in **tokens** (roughly ¾ of a word each).

Picture a small office desk. You can spread out maybe 20 documents. Bring a 21st, and something falls off the edge. Long conversations eventually overflow, and they cost more money too, since you pay per token.

Two classic solutions:

1. **Trimming (sliding window)** — keep only the last N messages. Simple, fast, but old details vanish.
2. **Summarisation** — periodically compress old turns into a short recap: *"User is Amaka, a fintech founder in Lagos, researching payment APIs."* Slower, but preserves the gist.

### What You'll Build Today

A memory layer that stores messages, tags them with a session ID, replays them on each turn, and gracefully handles conversations that grow too long.

You're about to turn a clever stranger into a genuine colleague. Let's code. 🚀


## ⚙️ 1. Setup

First, let's install the necessary libraries and configure your Google API key.


In [ ]:
# Install the Google Agent Development Kit (ADK)
!pip install -q google-adk

import os
import asyncio
from google.colab import userdata

# --- 🔑 Add your API key to the Colab Secrets Manager --- 
# 1. Click on the key icon in the left sidebar (Secrets).
# 2. Create a new secret with the name "GOOGLE_API_KEY".
# 3. Paste your API key as the value.
# 4. Make sure the toggle for "Notebook access" is on.

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("✅ GOOGLE_API_KEY configured successfully!")
except Exception as e:
    print(f"❌ Error configuring API key. Please follow the instructions above. Details: {e}")

# Import the ADK components we'll need
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService, preload_memory
from google.adk.runners import Runner
from google.adk.tools.tool_context import ToolContext
from google.genai import types


## 🧠 2. Short-Term Memory: Sessions

A **Session** is how the ADK tracks a single, continuous conversation. Think of it as one chat window. Everything the user says and the agent replies is recorded as an `Event` within that session.

- **Session**: A container for one conversation 💬
- **Events**: The individual messages and actions within that conversation 📝

By keeping track of all the events, the agent can "remember" what was said earlier in the *same* conversation. Let's see it in action!


In [ ]:
# A helper function to make running conversations easier
async def run_conversation(runner, session_id, messages):
    print(f"\n--- Starting Session: {session_id} ---")
    for msg in messages:
        print(f"\nUser: {msg}")
        response_text = ""
        async for event in runner.run_async(
            session_id=session_id,
            new_message=types.Content(parts=[types.Part(text=msg)])
        ):
            if event.content and event.content.parts:
                response_text += event.content.parts[0].text
        print(f"Agent: {response_text}")
    print("-----------------------------------")

# 1. Define a simple agent
chat_agent = LlmAgent(model=Gemini(), instruction="You are a friendly assistant.")

# 2. Use an in-memory session service (stores chats temporarily)
session_service = InMemorySessionService()

# 3. Create a Runner to manage the conversation
# We give it a unique user_id to identify who is talking.
runner = Runner(
    agent=chat_agent, 
    session_service=session_service, 
    user_id="beginner_user"
)

# 4. Have a multi-turn conversation
conversation = [
    "Hi, my name is Alex.",
    "What is my name?"
]

# Run the conversation in a single session
await run_conversation(runner, session_id="session-1", messages=conversation)

# The agent remembers the name because both messages are in the same session!


## ✍️ 3. Manual Memory: Session State

What if we want to store specific, structured information, not just the whole chat history? For this, we can use **Session State**.

`Session State` is like a digital sticky note or a Python dictionary (`{}`) attached to the conversation. Our agent can use **Tools** to write down and read back important facts, like a user's name, their favorite color, or an order number.

This is a form of *manual* memory management, where we explicitly tell the agent what to remember.


In [ ]:
# Tool to save a user's preference to the session's 'sticky note' (state)
def save_preference(tool_context: ToolContext, preference: str, value: str):
    """Saves a user's preference, like 'name' or 'favorite_color'."""
    key = f"user_preference_{preference}"
    tool_context.state[key] = value
    return f"OK, I've noted that the user's {preference} is {value}."

# Tool to retrieve a user's preference
def get_preference(tool_context: ToolContext, preference: str):
    """Gets a user's preference that was saved earlier."""
    key = f"user_preference_{preference}"
    value = tool_context.state.get(key, "I don't have a note about that.")
    return f"The user's {preference} is {value}."

# Create an agent that knows how to use these tools
stateful_agent = LlmAgent(
    model=Gemini(),
    instruction="You are a helpful assistant. Use your tools to save and get user preferences.",
    tools=[save_preference, get_preference]
)

# Create a new runner for this agent
stateful_runner = Runner(
    agent=stateful_agent, 
    session_service=session_service, # We can reuse the same session service
    user_id="beginner_user"
)

# Have a conversation where the agent uses its tools
stateful_conversation = [
    "Please remember my favorite color is sky blue.",
    "What is my favorite color?"
]

await run_conversation(stateful_runner, session_id="session-2", messages=stateful_conversation)

# In the first turn, the agent calls save_preference().
# In the second turn, it calls get_preference() to retrieve the data from the session state.


## 📚 4. Long-Term Memory: Automatic Memory

Sessions and State are great, but they are temporary. Once you start a *new* session, the agent forgets everything from the last one.

To solve this, the ADK has a **Memory** system for long-term storage. This is like a permanent, searchable database of important facts about a user.

- **Session (Short-Term):** Remembers things from the *current* conversation.
- **Memory (Long-Term):** Remembers things from *all past* conversations.

Best of all, we can automate it! We can tell the agent to **automatically save important facts** at the end of every conversation and **automatically load relevant memories** at the start of a new one.


In [ ]:
# This callback function will run automatically after every agent turn
async def auto_save_to_memory(callback_context):
    """A function that automatically saves the conversation to long-term memory."""
    print("[Auto-saving conversation to long-term memory...]")
    session = callback_context._invocation_context.session
    memory_service = callback_context._invocation_context.memory_service
    await memory_service.add_session_to_memory(session)

# 1. Create a long-term memory service
memory_service = InMemoryMemoryService()

# 2. Create an agent with automated memory
auto_memory_agent = LlmAgent(
    model=Gemini(),
    instruction="You are a personal assistant with a perfect memory.",
    # `preload_memory` automatically loads relevant memories for every new message.
    tools=[preload_memory],
    # `after_agent_callback` runs our auto_save_to_memory function after each turn.
    after_agent_callback=auto_save_to_memory
)

# 3. Create a runner with both session and memory services
auto_runner = Runner(
    agent=auto_memory_agent,
    session_service=session_service,
    memory_service=memory_service,
    user_id="beginner_user"
)

# --- Conversation 1 --- 
# The user states a preference. This will be auto-saved to long-term memory.
await run_conversation(auto_runner, session_id="long-term-session-1", messages=["I prefer all summaries to be in bullet points."])

# --- Conversation 2 (A completely new session!) ---
# The agent should remember the preference from the previous session.
await run_conversation(auto_runner, session_id="long-term-session-2", messages=["Can you summarize the plot of 'Dune'?"])


## 🚀 5. Mini-Capstone: Upgrade Your Universal Knowledge Worker

It's time to put it all together! Let's upgrade the Universal Knowledge Worker from Day 2. 

Your goal is to give the assistant a **persistent, long-term memory** so it can remember your research preferences across different chat sessions. For example, if you tell it, "From now on, always summarize in a professional tone," it should remember and apply that preference in all future research tasks.


In [ ]:
# We need a tool for the assistant. Let's reuse a simplified search tool.
# In a real scenario, this would be the powerful search tool from Day 2.
def search_the_web(query: str):
    """A simplified tool to search the web. Returns a dummy summary."""
    print(f"[Tool Called: Searching for '{query}']")
    return f"Summary for '{query}': The topic is complex with many recent developments. Key areas include X, Y, and Z."

# 1. Create the Research Assistant Agent with Automated Memory
# We'll reuse the `auto_save_to_memory` callback from the previous step.
research_assistant_agent = LlmAgent(
    model=Gemini(),
    instruction="You are a world-class research assistant. You must use the search_the_web tool to answer user queries. Pay close attention to the user's preferences for how to format your answers, which will be loaded into your memory.",
    tools=[
        search_the_web, # The agent's primary tool
        preload_memory    # The agent's memory tool
    ],
    after_agent_callback=auto_save_to_memory # The agent's auto-save mechanism
)

# 2. Create a new memory service just for our assistant
research_memory_service = InMemoryMemoryService()
research_session_service = InMemorySessionService()

# 3. Create the final runner
research_runner = Runner(
    agent=research_assistant_agent,
    session_service=research_session_service,
    memory_service=research_memory_service,
    user_id="research_user"
)

# --- SESSION 1: Set a preference ---
# The user tells the assistant how they want summaries formatted.
# This preference will be automatically saved to long-term memory.
await run_conversation(research_runner, session_id="research-session-1", messages=["For all future research, please summarize your findings in exactly three bullet points."])

# --- SESSION 2: Test the memory ---
# Start a new conversation and give a research task. The assistant should remember
# the formatting preference from the previous session.
await run_conversation(research_runner, session_id="research-session-2", messages=["Please research the latest advancements in quantum computing."])
